# Module 2.2 — Text Splitting Strategies

Splitting is crucial: chunks too large → irrelevant context; too small → incomplete answers.

## Splitters Covered
1. `RecursiveCharacterTextSplitter` — **recommended default**
2. `CharacterTextSplitter`
3. `TokenTextSplitter`
4. `SemanticChunker`
5. `MarkdownTextSplitter`
6. Code splitters
7. LLM-based splitting

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
    MarkdownTextSplitter,
    Language,
    RecursiveCharacterTextSplitter as CodeSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

SAMPLE_TEXT = """
Retrieval-Augmented Generation (RAG) is a technique that combines the power of large language
models with external knowledge retrieval. It was introduced to overcome two fundamental
limitations of LLMs: their static knowledge frozen at training time, and their tendency to
hallucinate facts.

The RAG pipeline has three core stages. First, documents are loaded and split into chunks.
Second, chunks are embedded into a vector space and stored in a vector database. Third, at
inference time, the user query is embedded and the most similar chunks are retrieved, then
passed to the LLM alongside the query.

This approach allows developers to build grounded, up-to-date, domain-specific AI assistants
without the expense of fine-tuning or the risk of hallucination inherent to pure generative
approaches.
"""


## 2.2.1 — RecursiveCharacterTextSplitter

Tries to split on `['\\n\\n', '\\n', ' ', '']` in order — preserving semantic units.

In [ ]:
# ── RecursiveCharacterTextSplitter ───────────────────────────────────────────
rcts = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # hierarchical separators
)

chunks = rcts.split_text(SAMPLE_TEXT)
print(f"RecursiveCharacterTextSplitter → {len(chunks)} chunks")
for i, c in enumerate(chunks, 1):
    print(f"  [{i}] ({len(c)} chars) {c[:80].strip()}...")


## 2.2.2 — CharacterTextSplitter & TokenTextSplitter

In [ ]:
# ── CharacterTextSplitter ────────────────────────────────────────────────────
cts = CharacterTextSplitter(separator="\n", chunk_size=200, chunk_overlap=20)
char_chunks = cts.split_text(SAMPLE_TEXT)
print(f"CharacterTextSplitter    → {len(char_chunks)} chunks")

# ── TokenTextSplitter ─────────────────────────────────────────────────────────
# splits by token count (tiktoken) rather than characters
tts = TokenTextSplitter(chunk_size=80, chunk_overlap=10)
tok_chunks = tts.split_text(SAMPLE_TEXT)
print(f"TokenTextSplitter        → {len(tok_chunks)} chunks")


## 2.2.3 — SemanticChunker

Uses embedding similarity between consecutive sentences to detect natural break-points.

Breakpoint types:
- `percentile` — split when distance > Nth percentile
- `standard_deviation` — split when distance > mean + N·std
- `gradient` — derivative of distances (good for longer texts)

In [ ]:
# ── SemanticChunker ───────────────────────────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

for bp_type in ["percentile", "standard_deviation", "gradient"]:
    chunker = SemanticChunker(embeddings, breakpoint_threshold_type=bp_type)
    sem_chunks = chunker.split_text(SAMPLE_TEXT)
    print(f"  SemanticChunker ({bp_type:<20}) → {len(sem_chunks)} chunks")


## 2.2.4 — MarkdownTextSplitter & Code Splitter

In [ ]:
# ── MarkdownTextSplitter ─────────────────────────────────────────────────────
MARKDOWN = """
# Introduction
This is the intro section explaining RAG basics.

## What is RAG?
RAG stands for Retrieval-Augmented Generation.

### Components
- Retriever
- Vector Store
- Generator

## How It Works
1. Embed query
2. Search vector store
3. Inject context into prompt
4. Generate answer
"""

md_splitter = MarkdownTextSplitter(chunk_size=150, chunk_overlap=20)
md_chunks   = md_splitter.split_text(MARKDOWN)
print(f"MarkdownTextSplitter → {len(md_chunks)} chunks")
for c in md_chunks:
    print(f"  {repr(c[:60])}...")

# ── Python Code Splitter ──────────────────────────────────────────────────────
PYTHON_CODE = """
def embed_query(query: str) -> list[float]:
    """Embed a query using OpenAI embeddings."""
    from openai import OpenAI
    client = OpenAI()
    return client.embeddings.create(input=query, model="text-embedding-3-small").data[0].embedding

def retrieve(query: str, vectorstore, k: int = 5):
    """Retrieve top-k relevant documents."""
    return vectorstore.similarity_search(query, k=k)

class RAGPipeline:
    def __init__(self, vectorstore, llm):
        self.vectorstore = vectorstore
        self.llm = llm

    def run(self, query: str) -> str:
        docs    = retrieve(query, self.vectorstore)
        context = "\\n".join(d.page_content for d in docs)
        return self.llm.invoke(f"Context: {context}\\nQuestion: {query}")
"""

code_splitter = CodeSplitter.from_language(
    language=Language.PYTHON, chunk_size=200, chunk_overlap=30
)
code_chunks = code_splitter.split_text(PYTHON_CODE)
print(f"\nPython CodeSplitter → {len(code_chunks)} chunks")
for c in code_chunks:
    print(f"  {repr(c[:70])}...")
